# Network Flow V2 — X-IIoTID: Step 02 / Feature and Data Quality (CORRECTED)

**Input:** `X-IIoTID dataset.csv` in `data/raw/x_iiotid/`. Run **Step 01** first.

**Correct X-IIoTID label structure (verified in your Step 01 output):**
- `class3`: **binary** Normal vs Attack.
- `class2`: broader attack category / attack stage (10 classes including Normal).
- `class1`: detailed attack type (19 classes including Normal).

This notebook performs a **read-only, chunk-based** audit (no cleaning or splitting): exact and feature-only duplicate signature counts, conflicts, numeric/placeholder checks, and correct label consistency across all three label levels. It saves a JSON report and a draft feature dictionary.

**How to run:** In VS Code choose the V2 Python kernel, then **Restart Kernel → Run All**. Cells depend on previous cells, particularly the duplicate-count cell. Use only the corrected notebook, not the previous Step 02 notebook.

**Caveats:** 64-bit fingerprint comparisons are screening evidence rather than exact cryptographic duplicate proofs. The dataset includes network and host telemetry: the feature dictionary is not a selected deployment-ready feature list.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from IPython.display import display
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "x_iiotid" / 'X-IIoTID dataset.csv'
AUDIT_PATH = ROOT / "reports" / "audits" / "x_iiotid_audit_v1.json"
REPORT_PATH = ROOT / "reports" / "audits" / "x_iiotid_feature_quality_v1.json"
DOC_PATH = ROOT / "docs" / "X_IIOTID_FEATURE_DICTIONARY_DRAFT.md"
CHUNK_SIZE = 12000
DISTINCT_CAP = 10000

if not AUDIT_PATH.is_file():
    raise FileNotFoundError("Run Step 01 first: " + str(AUDIT_PATH))
if not DATA.is_file():
    raise FileNotFoundError("Dataset not found: " + str(DATA))
with AUDIT_PATH.open("r", encoding="utf-8") as f:
    audit = json.load(f)

LABEL_COLUMNS = audit["candidate_label_columns"]
COLUMNS = audit["column_names"]
FEATURES = [c for c in COLUMNS if c not in LABEL_COLUMNS]
print("Dataset:", DATA.name)
print("Columns:", len(COLUMNS), "| candidate labels:", LABEL_COLUMNS)
print("Features inspected:", len(FEATURES))
print("Label meanings still require source-documentation verification.")


## 1. Confirm checksum and set the **correct** three label levels

For **X-IIoTID**, `class3` is the binary Normal/Attack target; `class2` is the broader category; `class1` is the detailed attack type. None of these three columns may be included in model input features. The labels are checked here for internal consistency only; the original documentation must still be consulted for semantic definitions.


In [ ]:
digest = hashlib.sha256()
with DATA.open("rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(block)
if digest.hexdigest() != audit["sha256"]:
    raise ValueError("CSV changed since Step 01 — rerun Step 01 before Step 02")

EXPECTED_LABELS = ("class1", "class2", "class3")
if not all(c in LABEL_COLUMNS for c in EXPECTED_LABELS):
    raise ValueError(f"Expected X-IIoTID labels {EXPECTED_LABELS}; found {LABEL_COLUMNS}")

binary_col = "class3"
category_col = "class2"
detailed_col = "class1"
print("Binary Normal/Attack label:", binary_col)
print("Broad attack category label:", category_col)
print("Detailed attack type label:", detailed_col)
print("Labels are ground truth only: exclude all three from model X.")


## 2. Full feature quality & duplicate screen

Values are read as **text** consistently across chunks to avoid inconsistent type-based hashing. Zero frequency means numeric-zero-looking values divided by all rows; it does **not** prove absence of a protocol. Exact-value cardinalities are capped at 10,000 for memory safety.

In [ ]:
MISSING_TOKENS = {"", "?", "na", "n/a", "nan", "none", "null", "undefined", "-", "inf", "-inf"}
missing = Counter()
placeholders = Counter()
zeros = Counter()
parseable = Counter()
nonnull = Counter()
distinct = {c: set() for c in FEATURES}
distinct_capped = set()
full_hash_chunks = []
feature_hash_chunks = []
label_hash_chunks = {c: [] for c in LABEL_COLUMNS}
row_count = 0

for chunk_number, part in enumerate(pd.read_csv(DATA, chunksize=CHUNK_SIZE,
                                              dtype="string", low_memory=False), start=1):
    if part.columns.tolist() != COLUMNS:
        raise ValueError("Column names differ from Step 01 audit")
    n = len(part)
    row_count += n
    # Stable 64-bit fingerprints: same raw string cells, same column order, same hash.
    full_hash_chunks.append(pd.util.hash_pandas_object(
        part[COLUMNS].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))
    feature_hash_chunks.append(pd.util.hash_pandas_object(
        part[FEATURES].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))
    for label in LABEL_COLUMNS:
        label_hash_chunks[label].append(pd.util.hash_pandas_object(
            part[[label]].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))

    for col in FEATURES:
        s = part[col]
        missing[col] += int(s.isna().sum())
        nonnull[col] += int(s.notna().sum())
        norm = s.str.strip()
        placeholders[col] += int(norm.str.casefold().isin(MISSING_TOKENS).sum())
        numeric = pd.to_numeric(norm, errors="coerce")
        parseable[col] += int(numeric.notna().sum())
        zeros[col] += int((numeric == 0).sum())
        if col not in distinct_capped:
            # Capped but accurate below the cap, even if early unique values repeat.
            for value in norm.dropna().unique():
                distinct[col].add(value)
                if len(distinct[col]) > DISTINCT_CAP:
                    distinct_capped.add(col)
                    distinct[col].clear()
                    break
    if chunk_number % 10 == 0:
        print(f"Processed {row_count:,} rows ({chunk_number} chunks)")

print(f"Full feature-quality scan complete: {row_count:,} rows")
if row_count != int(audit["row_count"]):
    raise ValueError("Row count differs from Step 01")


## 3. Duplicate records and feature-label conflicts — REQUIRED CELL

**Run this cell before the save cell.** It defines the duplicate statistics needed by the final JSON. Duplicates in a prepared ML CSV may be genuine repeated events or reused samples; do not automatically drop or split them across train/test. Group clashes are counted for each label level.


In [ ]:
full_hash = np.concatenate(full_hash_chunks)
feature_hash = np.concatenate(feature_hash_chunks)
_, full_count = np.unique(full_hash, return_counts=True)
_, feature_count = np.unique(feature_hash, return_counts=True)

exact_rows_in_duplicate_groups = int(full_count[full_count > 1].sum())
excess_full_duplicates = int((full_count[full_count > 1] - 1).sum())
feature_rows_in_duplicate_groups = int(feature_count[feature_count > 1].sum())
excess_feature_duplicates = int((feature_count[feature_count > 1] - 1).sum())

print("Rows in identical full-record groups:", exact_rows_in_duplicate_groups)
print("Excess identical rows:", excess_full_duplicates)
print("Rows in identical candidate-feature groups:", feature_rows_in_duplicate_groups)
print("Excess candidate-feature duplicates:", excess_feature_duplicates)

label_conflicts = {}
for label in LABEL_COLUMNS:
    hashes = np.concatenate(label_hash_chunks[label])
    ordered = np.lexsort((hashes, feature_hash))
    sorted_fp = feature_hash[ordered]
    sorted_label = hashes[ordered]
    conflicting_boundary = (sorted_fp[1:] == sorted_fp[:-1]) & (sorted_label[1:] != sorted_label[:-1])
    conflicting_groups = np.unique(sorted_fp[:-1][conflicting_boundary])
    label_conflicts[label] = int(len(conflicting_groups))
    print(f"Feature groups with differing {label!r} labels:", label_conflicts[label])

del full_hash_chunks, feature_hash_chunks, label_hash_chunks


## 4. Zero-heavy, constant, mixed-data and possible leakage fields

This prepares a review table. **Do not automatically remove zero-heavy fields**; IoT traffic naturally contains protocol-specific zeros. Constant features have no variance in this particular CSV, but feature removal for a deployed classifier should be validated against future traffic and training splits.

In [ ]:
quality_rows=[]
for col in FEATURES:
    is_capped = col in distinct_capped
    q={
        "feature":col,
        "missing_count":int(missing[col]),
        "placeholder_token_count":int(placeholders[col]),
        "numeric_parse_pct":round(100 * parseable[col] / max(nonnull[col],1), 2),
        "numeric_zero_pct_of_all_rows":round(100 * zeros[col] / max(row_count,1), 2),
        "distinct_values":None if is_capped else int(len(distinct[col])),
        "distinct_count_at_least":DISTINCT_CAP + 1 if is_capped else int(len(distinct[col])),
        "distinct_capped":is_capped,
        "review_as_identifier_or_payload":any(token in col.casefold() for token in (
            "ip.", "src_ip", "dst_ip", "addr", "id", "time", "timestamp", "payload", "content", "uri", "msg", "session")),
    }
    quality_rows.append(q)
quality = pd.DataFrame(quality_rows)
print("Potential constant non-label fields:")
display(quality[(quality.distinct_values == 1)][["feature", "distinct_values"]])
print("Features with >=95% numeric zero values (investigate):")
display(quality[quality.numeric_zero_pct_of_all_rows >= 95]
        [["feature", "numeric_zero_pct_of_all_rows", "distinct_count_at_least"]]
        .sort_values("numeric_zero_pct_of_all_rows", ascending=False).head(35))
print("Missing/placeholder markers:")
display(quality.sort_values(["missing_count","placeholder_token_count"], ascending=False)
        [["feature", "missing_count", "placeholder_token_count"]].head(25))
print("Possible identifiers, time, payload or text features needing leakage/privacy review:")
display(quality[quality.review_as_identifier_or_payload][["feature", "numeric_parse_pct", "distinct_count_at_least"]].head(45))


## 5. Check **binary vs both attack-type label levels**

Correct mapping: `class3` is binary Normal/Attack; both `class2` and `class1` use Normal for benign observations. This check verifies only normal/attack agreement, **not** that individual attack-stage and attack-type mappings are correct. Any missing or unrecognized binary label is counted as uncheckable.


In [ ]:
def binary_target(s):
    txt = s.astype("string").str.strip().str.casefold()
    return txt.map({"0": 0, "1": 1, "normal": 0, "benign": 0,
                    "attack": 1, "malicious": 1})

def category_consistency(y, categories):
    kind = categories.astype("string").str.strip().str.casefold()
    valid = kind.notna() & kind.ne("") & (~kind.isin(MISSING_TOKENS))
    compare = y.notna() & valid
    is_normal = kind.isin(["normal", "benign"])
    mismatch = ((y == 0) != is_normal).fillna(False) & compare
    return int(compare.sum()), int(mismatch.sum()), int((~compare).sum())

label_checked = 0
label_mismatch = 0
label_unknown = 0
detail_checked = 0
detail_mismatch = 0
detail_unknown = 0
binary_distribution = Counter()

for part in pd.read_csv(DATA, chunksize=CHUNK_SIZE, dtype="string", low_memory=False,
                        usecols=[binary_col, category_col, detailed_col]):
    y = binary_target(part[binary_col])
    for key, count in part[binary_col].fillna("<missing>").value_counts().items():
        binary_distribution[str(key)] += int(count)

    a, b, c = category_consistency(y, part[category_col])
    label_checked += a
    label_mismatch += b
    label_unknown += c

    a, b, c = category_consistency(y, part[detailed_col])
    detail_checked += a
    detail_mismatch += b
    detail_unknown += c

print("Binary class3 counts:", dict(binary_distribution))
print("class3 vs class2 — checked:", label_checked,
      "| mismatches:", label_mismatch,
      "| uncheckable:", label_unknown)
print("class3 vs class1 — checked:", detail_checked,
      "| mismatches:", detail_mismatch,
      "| uncheckable:", detail_unknown)
if label_mismatch or detail_mismatch:
    print("WARNING: label inconsistencies found. Investigate before splitting.")
if label_unknown or detail_unknown:
    print("WARNING: some rows could not be checked. Investigate original values.")


## 6. Save reports and a *draft* feature dictionary

These are descriptive outputs, not final feature selection. Physical meaning, units, scope (network/host/sensor), timing and live-gateway feasibility still need source verification.

In [ ]:
required = ("exact_rows_in_duplicate_groups", "excess_full_duplicates",
            "feature_rows_in_duplicate_groups", "excess_feature_duplicates", "label_conflicts")
missing_variables = [name for name in required if name not in globals()]
if missing_variables:
    raise RuntimeError("Required duplicate-analysis variables missing: " + ", ".join(missing_variables) +
                       ". Restart the kernel and Run All, including the duplicate-count cell.")

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
DOC_PATH.parent.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset":'X-IIoTID',
    "filename":DATA.name,
    "sha256":audit["sha256"],
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "row_count":row_count,
    "candidate_labels":LABEL_COLUMNS,
    "provisional_binary_label":binary_col,
    "detailed_attack_type_label":detailed_col,
    "provisional_category_label":category_col,
    "exact_duplicate_rows_in_groups":exact_rows_in_duplicate_groups,
    "excess_exact_duplicates":excess_full_duplicates,
    "feature_duplicate_rows_in_groups":feature_rows_in_duplicate_groups,
    "excess_feature_duplicates":excess_feature_duplicates,
    "feature_fingerprint_label_conflict_groups":label_conflicts,
    "binary_category_rows_checked":label_checked,
    "binary_category_mismatches":label_mismatch,
    "binary_category_uncheckable":label_unknown,
    "binary_detailed_rows_checked":detail_checked,
    "binary_detailed_mismatches":detail_mismatch,
    "binary_detailed_uncheckable":detail_unknown,
    "binary_label_distribution":dict(binary_distribution),
    "feature_statistics":quality_rows,
    "caveats":[
        "Hashes are noncryptographic and duplicate counts are screening statistics.",
        "Constant and sparse features must not automatically be dropped before a split decision.",
        "Input values were compared lexically after CSV parsing; raw canonicalization is not established.",
        "Feature meanings/units/time basis are not yet verified from original documentation.",
        "No train/validation/test partitions were created.",
        "class3=Normal/Attack, class2=broad category, class1=detailed attack type."
    ]
}
with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

lines=[
    f"# X-IIoTID — Feature Dictionary DRAFT",
    "",
    f"Source: `X-IIoTID dataset.csv`; SHA-256: `{audit['sha256']}`",
    "",
    "**Unverified columns:** meaning, units, source modality, temporal window, and deployability must be documented manually.",
    "",
    "| Feature | Numeric parse % | Numeric zeros % | Distinct >= | Missing | Placeholders | Candidate role |",
    "|---|---:|---:|---:|---:|---:|---|",
]
for q in quality_rows:
    name = str(q['feature']).replace('|', chr(92)+'|')
    role = "REVIEW: identifier/time/payload" if q['review_as_identifier_or_payload'] else "VERIFY modality/units"
    lines.append(f"| {name} | {q['numeric_parse_pct']} | {q['numeric_zero_pct_of_all_rows']} | {q['distinct_count_at_least']}{'+' if q['distinct_capped'] else ''} | {q['missing_count']} | {q['placeholder_token_count']} | {role} |")
DOC_PATH.write_text("\n".join(lines)+"\n", encoding="utf-8")
print("Saved feature-quality report:",REPORT_PATH)
print("Saved draft feature dictionary:",DOC_PATH)


## What to share

Please send **Step 02 corrected output** containing:

1. `Rows in identical full-record groups` and `Excess identical rows`.
2. All three `Feature groups with differing ... labels` counts.
3. `class3 vs class2` and `class3 vs class1` **checked / mismatches / uncheckable**.
4. The two saved report paths.

Then we can prepare the three-dataset feature mapping without premature normalization or train/test splitting.
